# DS317.R11 – Giai đoạn 2: Đo chất lượng dữ liệu (trước tiền xử lý)

Đề tài: Dự báo doanh thu 4 tuần theo danh mục cho doanh nghiệp thời trang thương mại điện tử (dữ liệu Datathon 2026).

Notebook chấm điểm 13 bảng theo 6 khía cạnh của Chương 4: Đầy đủ, Hợp lệ, Chính xác, Nhất quán, Duy nhất, Kịp thời. Kết quả là mốc **"trước"** để so với mốc "sau" ở Giai đoạn 3. Cách chạy: điền `DROP_COLS` ở Phần 0, rồi chạy **Restart and run all**.

**Công thức.** Điểm 1 quy tắc = 1 − số dòng vi phạm / số dòng kiểm tra (slide 16). Điểm 1 bảng ở 1 khía cạnh = trung bình điểm các quy tắc thuộc khía cạnh đó (slide 17).

**Phân loại vi phạm.** Mỗi quy tắc có vi phạm được xếp vào một trong hai loại:
- **lỗi thật**: dữ liệu sai hoặc thiếu, có tính điểm;
- **đúng nghiệp vụ**: trông lạ nhưng phản ánh hoạt động thật, giữ số Kiểm tra / Vi phạm để báo cáo nhưng không tính điểm.

| Phần | Nội dung | Dùng cho |
| --- | --- | --- |
| 0 | Cấu hình, cột đã drop, hàm ghi quy tắc | – |
| 1 | Đọc dữ liệu, bảng nối dùng chung | – |
| 2 | sales, promotions; kiểm tra Urban Blowout | Bảng điểm, mục lỗi khuyến mãi |
| 3 | orders, order_items | Bảng điểm |
| 4 | products, inventory | Bảng điểm |
| 5 | web_traffic, payments | Bảng điểm, khoảng thiếu traffic |
| 6 | shipments, returns, reviews | Bảng điểm, đơn thiếu vận đơn |
| 7 | customers, geography | Bảng điểm, `signup_date` |
| 8 | Bảng điểm "trước", danh sách lỗi thật, danh sách đúng nghiệp vụ, bảng "Không đo" | Báo cáo GĐ2 |
| 9 | Điểm theo năm | Màn hình theo dõi theo thời gian |
| 10 | Lưu kết quả ra `outputs/cldl/` | Giai đoạn 3 |

**Không làm** (ghi trong báo cáo kèm lý do):
- AccTEMPO và mặt phẳng chẩn đoán: dành cho bài phân loại nhiều lớp có mốc dự đoán sớm, còn đề tài là hồi quy doanh thu.
- 15 đặc tính ISO/IEC 25012: dùng bộ 6 khía cạnh cốt lõi là đủ.
- Đánh giá chủ quan: không có người dùng dữ liệu thật để khảo sát.

**Sửa so với notebook `eda/06` cũ:** không trừ điểm ô trống mang nghĩa "không áp dụng" (`promo_id`, `promo_id_2`, `applicable_category`); không coi `unit_price` khác `products.price` là lỗi Chính xác; không coi 382 ngày lỗ gộp là lỗi.

In [ ]:
import sys
from pathlib import Path

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src" / "ds317").is_dir())
sys.path.insert(0, str(ROOT / "src"))

import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display

from ds317 import load
from ds317.paths import out_dir

%matplotlib inline
warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 50)
pd.set_option("display.max_colwidth", 80)
OUT = out_dir("cldl")

# Phần 0. Cấu hình

`DROP_COLS`: cột đã quyết định không dùng cho đầu ra, kèm lý do. Dạng `{"tên_bảng": {"tên_cột": "lý do"}}`.

Lưu ý: đừng drop cột dùng để nối hoặc lọc bảng (`order_id`, `product_id`, `order_date`, `order_status`, `category`), dù nó không làm feature.

In [ ]:
DROP_COLS = {
    # Ví dụ:
    # "customers": {"gender": "không dùng cho dự báo doanh thu tuần x danh mục"},
    # "reviews": {"review_title": "văn bản tự do, không dùng"},
}

DIMS = ["Đầy đủ", "Hợp lệ", "Chính xác", "Nhất quán", "Duy nhất", "Kịp thời"]
RULES, BY_YEAR, EXAMPLES, SKIPPED = [], [], {}, []


def chk(bang, cot, khia_canh, quy_tac, bad, df=None, nam=None, phan_loai=None, xu_ly=""):
    # Ghi 1 quy tắc. bad = Series bool, True là dòng vi phạm, mỗi phần tử là 1 dòng được kiểm tra.
    cots = [cot] if isinstance(cot, str) else list(cot)
    bo = [c for c in cots if c in DROP_COLS.get(bang, {})]
    if bo:
        SKIPPED.append({"Bảng": bang, "Cột": ", ".join(cots), "Quy tắc": quy_tac,
                        "Lý do": "; ".join(f"{c}: {DROP_COLS[bang][c]}" for c in bo)})
        return
    bad = pd.Series(np.asarray(bad)).fillna(False).astype(bool)
    n, v = len(bad), int(bad.sum())
    rid = f"R{len(RULES) + 1:02d}"
    if phan_loai is None:
        phan_loai = "đạt" if v == 0 else "cần xem"
    diem = round(1 - v / n, 3) if n else np.nan
    RULES.append({"ID": rid, "Bảng": bang, "Cột": ", ".join(cots), "Khía cạnh": khia_canh,
                  "Quy tắc": quy_tac, "Kiểm tra": n, "Vi phạm": v,
                  "Điểm": np.nan if phan_loai == "đúng nghiệp vụ" else diem,
                  "Phân loại": phan_loai, "Xử lý ở GĐ3": xu_ly})
    if v and df is not None:
        EXAMPLES[rid] = df[bad.values].head(3)
    if nam is not None:
        g = (pd.DataFrame({"nam": np.asarray(nam), "bad": bad.values})
             .groupby("nam")["bad"].agg(["size", "sum"]).reset_index())
        g["diem"] = 1 - g["sum"] / g["size"]
        g["ID"], g["Quy tắc"] = rid, f"{bang}: {quy_tac}"
        BY_YEAR.append(g)


def xem(*bang):
    # Bảng quy tắc của các bảng dữ liệu vừa đo
    r = pd.DataFrame(RULES)
    display(r[r["Bảng"].isin(bang)].fillna({"Điểm": "—"}))


def vd(rid):
    # 3 dòng vi phạm đầu tiên của 1 quy tắc
    return EXAMPLES.get(rid, "Không có dòng vi phạm (hoặc quy tắc không lưu ví dụ).")

# Phần 1. Đọc dữ liệu và bảng nối dùng chung

`lines` = `order_items` nối thêm ngày đặt, trạng thái đơn (từ `orders`), danh mục và giá vốn (từ `products`). Nhiều quy tắc ở các phần sau dùng chung bảng này.

In [ ]:
sales = load("sales"); promotions = load("promotions")
orders = load("orders"); order_items = load("order_items")
products = load("products"); inventory = load("inventory")
web_traffic = load("web_traffic"); payments = load("payments")
shipments = load("shipments"); returns = load("returns"); reviews = load("reviews")
customers = load("customers"); geography = load("geography")

# Dòng hàng kèm ngày đặt, trạng thái, danh mục, giá vốn
lines = (order_items
         .merge(orders[["order_id", "order_date", "order_status", "customer_id"]], on="order_id", how="left")
         .merge(products[["product_id", "category", "price", "cogs"]], on="product_id", how="left"))
lines["gross"] = lines["quantity"] * lines["unit_price"]

{k: len(v) for k, v in dict(sales=sales, promotions=promotions, orders=orders, order_items=order_items,
                           products=products, inventory=inventory, web_traffic=web_traffic, payments=payments,
                           shipments=shipments, returns=returns, reviews=reviews, customers=customers,
                           geography=geography).items()}

# Phần 2. sales, promotions

Kiểm tra:
1. Duy nhất: `sales.Date`, `promotions.promo_id`.
2. Đầy đủ: `sales` đủ mọi ngày 04/07/2012–31/12/2022; các cột chính không trống.
3. Hợp lệ: `Revenue`, `COGS` > 0; `start_date` ≤ `end_date`; mức giảm nằm trong miền cho phép.
4. Chính xác: `Revenue` mỗi ngày bằng tổng `quantity` × `unit_price` của `order_items` ngày đó.
5. Nhất quán: mã khuyến mãi trong `order_items` có thật, dùng đúng thời gian và đúng danh mục.

In [ ]:
# Duy nhất
chk("sales", "Date", "Duy nhất", "Date không trùng", sales["Date"].duplicated(keep=False), sales)
chk("promotions", "promo_id", "Duy nhất", "promo_id không trùng", promotions["promo_id"].duplicated(keep=False), promotions)

# Đầy đủ: đủ mọi ngày 04/07/2012 - 31/12/2022
all_days = pd.date_range("2012-07-04", "2022-12-31", freq="D")
chk("sales", "Date", "Đầy đủ", "đủ mọi ngày 04/07/2012-31/12/2022", ~all_days.isin(sales["Date"]))
chk("sales", ["Revenue", "COGS"], "Đầy đủ", "Revenue, COGS không trống",
    sales[["Revenue", "COGS"]].isna().any(axis=1), sales)
core_promo = ["promo_id", "promo_name", "promo_type", "discount_value", "start_date", "end_date", "promo_channel"]
chk("promotions", core_promo, "Đầy đủ", "các cột chính không trống (bỏ applicable_category)",
    promotions[core_promo].isna().any(axis=1), promotions)
chk("promotions", "applicable_category", "Đầy đủ", "applicable_category không trống",
    promotions["applicable_category"].isna(), promotions,
    phan_loai="đúng nghiệp vụ", xu_ly="trống = áp dụng mọi danh mục")

# Hợp lệ
chk("sales", "Revenue", "Hợp lệ", "Revenue > 0", sales["Revenue"] <= 0, sales)
chk("sales", "COGS", "Hợp lệ", "COGS > 0", sales["COGS"] <= 0, sales)
chk("promotions", ["start_date", "end_date"], "Hợp lệ", "start_date <= end_date",
    promotions["start_date"] > promotions["end_date"], promotions)
pct = promotions["promo_type"] == "percentage"
chk("promotions", "discount_value", "Hợp lệ", "percentage: discount_value thuộc (0; 100]",
    ~promotions.loc[pct, "discount_value"].between(0, 100, inclusive="right"), promotions[pct])
chk("promotions", "discount_value", "Hợp lệ", "fixed: discount_value > 0",
    promotions.loc[~pct, "discount_value"] <= 0, promotions[~pct])

# Chính xác: Revenue ngày = tổng quantity x unit_price của order_items ngày đó (gồm đơn huỷ)
daily = lines.groupby("order_date")["gross"].sum()
diff = sales["Revenue"] - sales["Date"].map(daily).fillna(0)
chk("sales", "Revenue", "Chính xác", "Revenue = tổng quantity x unit_price trong ngày (lệch > 1)",
    diff.abs() > 1, sales.assign(tinh_lai=sales["Date"].map(daily), lech=diff))

# Ghi riêng: ngày lỗ gộp
chk("sales", ["Revenue", "COGS"], "Hợp lệ", "COGS <= Revenue (ngày không lỗ gộp)",
    sales["COGS"] > sales["Revenue"], sales, nam=sales["Date"].dt.year,
    phan_loai="đúng nghiệp vụ", xu_ly="giữ, do khuyến mãi")

In [ ]:
# Nhất quán giữa order_items và promotions (xét cả promo_id và promo_id_2)
pl = pd.concat([lines[["order_id", "product_id", "order_date", "category", c]].rename(columns={c: "pid"})
                for c in ["promo_id", "promo_id_2"]]).dropna(subset=["pid"])
pl = pl.merge(promotions[["promo_id", "start_date", "end_date", "applicable_category"]],
              left_on="pid", right_on="promo_id", how="left")

chk("order_items", ["promo_id", "promo_id_2"], "Nhất quán", "mã khuyến mãi có trong promotions",
    pl["promo_id"].isna(), pl)
ok = pl["promo_id"].notna()
chk("order_items", ["promo_id", "promo_id_2"], "Nhất quán", "ngày đặt nằm trong start_date-end_date của mã",
    ~pl.loc[ok, "order_date"].between(pl.loc[ok, "start_date"], pl.loc[ok, "end_date"]), pl[ok])
cat = pl["applicable_category"].notna()
chk("order_items", ["promo_id", "promo_id_2"], "Nhất quán", "sản phẩm đúng applicable_category của mã",
    pl.loc[cat, "category"] != pl.loc[cat, "applicable_category"], pl[cat])

xem("sales", "promotions")

**Kiểm tra Urban Blowout (`fixed` 50):** mức giảm thật trên mỗi dòng là bao nhiêu? Nếu `giam_moi_sp` gần 50 thì 50 là số tiền giảm trên 1 sản phẩm. Nếu `ty_le_giam` gần 0,5 thì thực chất là giảm 50% và `promo_type` ghi sai.

In [ ]:
ub = promotions[promotions["promo_name"].str.contains("Urban Blowout")]
x = lines[lines["promo_id"].isin(ub["promo_id"]) | lines["promo_id_2"].isin(ub["promo_id"])].copy()
x["giam_moi_sp"] = x["discount_amount"] / x["quantity"]
x["ty_le_giam"] = x["discount_amount"] / x["gross"]
display(ub[["promo_id", "promo_name", "promo_type", "discount_value", "start_date", "end_date"]])
x[["discount_amount", "giam_moi_sp", "ty_le_giam"]].describe().round(3)

**Nhận xét**

- `sales` đạt điểm tối đa ở mọi khía cạnh: 3.833 ngày, không thiếu, không trùng, không âm.
- `Revenue` của `sales` **khớp tuyệt đối** với tổng `quantity` × `unit_price` của `order_items` (tính cả đơn cancelled), lệch 0 ở cả 3.833 ngày. Vì vậy có thể dựng doanh thu theo danh mục từ `order_items` mà vẫn cộng lại đúng tổng của công ty.
- 382 ngày `COGS` > `Revenue` (10%) là **đúng nghiệp vụ**: đây là các ngày lỗ gộp do khuyến mãi, không trừ điểm.
- `promotions`: 50 mã, không lỗi. 40/50 mã trống `applicable_category` là **đúng nghiệp vụ** (áp dụng mọi danh mục).
- 276.522 lượt dùng mã trong `order_items` đều có mã trong `promotions`, đều nằm trong thời gian chạy của mã, và đều đúng danh mục.
- **Urban Blowout `fixed` 50 là đúng.** `giam_moi_sp` có giá trị nhỏ nhất, Q1, trung vị, Q3 đều bằng 50, nên 50 là số tiền giảm trên 1 sản phẩm. 99,0% dòng giảm đúng 50/sp; 206 dòng giảm hơn 50/sp đều dùng thêm mã thứ 2 (`promo_id_2`). Không phải lỗi Chính xác.

# Phần 3. orders, order_items

Kiểm tra:
1. Duy nhất: `orders.order_id`; cặp (`order_id`, `product_id`) trong `order_items`.
2. Đầy đủ: cột khoá, ngày, tiền không trống. `promo_id`, `promo_id_2` trống là "không dùng khuyến mãi".
3. Hợp lệ: lượng, giá > 0; chiết khấu không vượt giá trị dòng; trạng thái thuộc danh sách; giá bán so với giá vốn.
4. Nhất quán: khoá nối giữa `order_items`, `orders`, `products`, `customers`.
5. Kịp thời: ngày nào của `sales` cũng có đơn.

In [ ]:
# Duy nhất
chk("orders", "order_id", "Duy nhất", "order_id không trùng", orders["order_id"].duplicated(keep=False), orders)
chk("order_items", ["order_id", "product_id"], "Duy nhất", "(order_id, product_id) không trùng",
    order_items.duplicated(["order_id", "product_id"], keep=False), order_items,
    phan_loai="đúng nghiệp vụ", xu_ly="giữ, là các dòng hàng tách riêng (Phần B)")

# Đầy đủ
oc = ["order_id", "order_date", "customer_id", "zip", "order_status", "payment_method"]
chk("orders", oc, "Đầy đủ", "cột khoá, ngày, trạng thái không trống", orders[oc].isna().any(axis=1), orders)
ic = ["order_id", "product_id", "quantity", "unit_price", "discount_amount"]
chk("order_items", ic, "Đầy đủ", "cột khoá, lượng, tiền không trống", order_items[ic].isna().any(axis=1), order_items)
chk("order_items", "promo_id", "Đầy đủ", "promo_id không trống", order_items["promo_id"].isna(),
    phan_loai="đúng nghiệp vụ", xu_ly="đổi thành cột có/không khuyến mãi")
chk("order_items", "promo_id_2", "Đầy đủ", "promo_id_2 không trống", order_items["promo_id_2"].isna(),
    phan_loai="đúng nghiệp vụ", xu_ly="trống = không dùng mã thứ 2")

# Hợp lệ
chk("order_items", "quantity", "Hợp lệ", "quantity > 0", order_items["quantity"] <= 0, order_items)
chk("order_items", "unit_price", "Hợp lệ", "unit_price > 0", order_items["unit_price"] <= 0, order_items)
chk("order_items", "discount_amount", "Hợp lệ", "0 <= discount_amount <= quantity x unit_price",
    (lines["discount_amount"] < 0) | (lines["discount_amount"] > lines["gross"] + 1e-6), lines)
chk("orders", "order_status", "Hợp lệ", "order_status thuộc danh sách",
    ~orders["order_status"].isin(["created", "paid", "shipped", "delivered", "returned", "cancelled"]), orders)
chk("order_items", "unit_price", "Hợp lệ", "unit_price >= cogs (không bán dưới giá vốn)",
    lines["unit_price"] < lines["cogs"], lines, nam=lines["order_date"].dt.year,
    phan_loai="đúng nghiệp vụ", xu_ly="giữ, 99,4% dòng vi phạm có khuyến mãi")

# Nhất quán
chk("order_items", "order_id", "Nhất quán", "order_id có trong orders",
    ~order_items["order_id"].isin(orders["order_id"]), order_items)
chk("order_items", "product_id", "Nhất quán", "product_id có trong products",
    ~order_items["product_id"].isin(products["product_id"]), order_items)
chk("orders", "customer_id", "Nhất quán", "customer_id có trong customers",
    ~orders["customer_id"].isin(customers["customer_id"]), orders)
chk("orders", "order_id", "Nhất quán", "đơn có ít nhất 1 dòng hàng",
    ~orders["order_id"].isin(order_items["order_id"]), orders)

# Kịp thời
chk("orders", "order_date", "Kịp thời", "mọi ngày của sales có ít nhất 1 đơn",
    ~sales["Date"].isin(orders["order_date"]), sales)

xem("orders", "order_items")

**Nhận xét**

- Không có lỗi thật ở `orders` và `order_items`: khoá không trùng, không trống, khoá nối đều tồn tại, ngày nào cũng có đơn.
- 32 dòng (16 cặp) trùng (`order_id`, `product_id`) nhưng khác `quantity`, `unit_price`. Đây là hai dòng hàng riêng của cùng một SKU trong một đơn (Phần B notebook đề xuất), nên **đúng nghiệp vụ**, giữ nguyên.
- `promo_id` trống 61,3%, `promo_id_2` trống 99,97%: **đúng nghiệp vụ**. Ở Giai đoạn 3 đổi thành cờ có/không khuyến mãi.
- 133.052 dòng (18,6%) bán dưới giá vốn. **99,4%** số dòng này có `promo_id`, còn dòng không khuyến mãi chỉ 808 dòng, giá bán quanh 94,5% giá niêm yết. Vì vậy xếp **đúng nghiệp vụ**: lỗ do khuyến mãi, không phải giá nhập sai.

# Phần 4. products, inventory

Kiểm tra:
1. Duy nhất: `products.product_id`; (`product_id`, `snapshot_date`) trong `inventory`.
2. Hợp lệ: giá, giá vốn > 0 và giá vốn < giá; danh mục thuộc 4 danh mục; các tỷ lệ thuộc [0; 1]; số ngày hết hàng không vượt số ngày của tháng.
3. Nhất quán: `inventory` khớp `products` về SKU và danh mục; hai cờ hết hàng và tồn dư.
4. Kịp thời: `inventory` đủ 126 kỳ cuối tháng.

In [ ]:
# Duy nhất
chk("products", "product_id", "Duy nhất", "product_id không trùng",
    products["product_id"].duplicated(keep=False), products)
chk("inventory", ["product_id", "snapshot_date"], "Duy nhất", "(product_id, snapshot_date) không trùng",
    inventory.duplicated(["product_id", "snapshot_date"], keep=False), inventory)

# Đầy đủ
chk("products", ["product_id", "category", "price", "cogs"], "Đầy đủ", "cột khoá, danh mục, giá không trống",
    products[["product_id", "category", "price", "cogs"]].isna().any(axis=1), products)

# Hợp lệ
chk("products", "price", "Hợp lệ", "price > 0", products["price"] <= 0, products)
chk("products", "cogs", "Hợp lệ", "cogs > 0", products["cogs"] <= 0, products)
chk("products", ["price", "cogs"], "Hợp lệ", "cogs < price", products["cogs"] >= products["price"], products)
chk("products", "category", "Hợp lệ", "category thuộc 4 danh mục",
    ~products["category"].isin(["Casual", "GenZ", "Outdoor", "Streetwear"]), products)
chk("products", "price", "Hợp lệ", "price >= 100", products["price"] < 100, products,
    phan_loai="đúng nghiệp vụ", xu_ly="giữ, SKU giá thấp")
chk("inventory", "stock_on_hand", "Hợp lệ", "stock_on_hand >= 0", inventory["stock_on_hand"] < 0, inventory)
for c in ["fill_rate", "sell_through_rate"]:
    chk("inventory", c, "Hợp lệ", f"{c} thuộc [0; 1]", ~inventory[c].between(0, 1), inventory)
chk("inventory", "stockout_days", "Hợp lệ", "stockout_days <= số ngày của tháng",
    inventory["stockout_days"] > inventory["snapshot_date"].dt.days_in_month, inventory)

# Nhất quán
chk("inventory", "product_id", "Nhất quán", "product_id có trong products",
    ~inventory["product_id"].isin(products["product_id"]), inventory)
chk("inventory", "category", "Nhất quán", "category khớp products",
    inventory["category"].values != inventory["product_id"].map(products.set_index("product_id")["category"]).values,
    inventory)
chk("inventory", ["stockout_flag", "overstock_flag"], "Nhất quán", "không vừa hết hàng vừa tồn dư",
    (inventory["stockout_flag"] == 1) & (inventory["overstock_flag"] == 1), inventory,
    phan_loai="đúng nghiệp vụ", xu_ly="giữ, hai cờ đo hai tiêu chí khác nhau")

# Kịp thời: đủ 126 kỳ cuối tháng
month_ends = pd.date_range("2012-07-31", "2022-12-31", freq="ME")
chk("inventory", "snapshot_date", "Kịp thời", "đủ 126 kỳ cuối tháng 07/2012-12/2022",
    ~month_ends.isin(inventory["snapshot_date"]))

xem("products", "inventory")

**Nhận xét**

- `products` không có lỗi thật. 654 SKU giá dưới 100 là **đúng nghiệp vụ** (SKU giá thấp có thật, Phần B notebook đề xuất).
- `inventory` đủ 126 kỳ cuối tháng, không trùng, các tỷ lệ và số ngày đều hợp lệ, SKU và danh mục khớp `products`.
- 30.495 snapshot (50,6%) vừa có cờ hết hàng vừa có cờ tồn dư. Hai cờ đo hai tiêu chí khác nhau: có ngày hết hàng trong tháng, và `days_of_supply` > 90 ở cuối tháng. Vì vậy xếp **đúng nghiệp vụ**, nhưng cặp cờ này ít giá trị khi làm đặc trưng.

# Phần 5. web_traffic, payments

Kiểm tra:
1. Duy nhất: ngày trong `web_traffic`; `payments.order_id`.
2. Hợp lệ: các cột đếm ≥ 0; `bounce_rate` thuộc [0; 1]; phương thức và số kỳ trả góp thuộc danh sách.
3. Nhất quán: `payments` khớp `orders` về mã đơn và phương thức thanh toán.
4. Chính xác: `payment_value` bằng tổng tiền hàng sau giảm của đơn.
5. Kịp thời: ngày nào của `sales` cũng có traffic.

In [ ]:
# Duy nhất
chk("web_traffic", "date", "Duy nhất", "date không trùng", web_traffic["date"].duplicated(keep=False), web_traffic)
chk("payments", "order_id", "Duy nhất", "order_id không trùng", payments["order_id"].duplicated(keep=False), payments)

# Hợp lệ
cnt = ["sessions", "unique_visitors", "page_views", "avg_session_duration_sec"]
chk("web_traffic", cnt, "Hợp lệ", "các cột đếm >= 0", (web_traffic[cnt] < 0).any(axis=1), web_traffic)
chk("web_traffic", "bounce_rate", "Hợp lệ", "bounce_rate thuộc [0; 1]", ~web_traffic["bounce_rate"].between(0, 1), web_traffic)
chk("web_traffic", ["unique_visitors", "sessions"], "Hợp lệ", "unique_visitors <= sessions",
    web_traffic["unique_visitors"] > web_traffic["sessions"], web_traffic)
chk("payments", "payment_value", "Hợp lệ", "payment_value > 0", payments["payment_value"] <= 0, payments)
methods = ["apple_pay", "bank_transfer", "cod", "credit_card", "paypal"]
chk("payments", "payment_method", "Hợp lệ", "payment_method thuộc danh sách",
    ~payments["payment_method"].isin(methods), payments)
chk("payments", "installments", "Hợp lệ", "installments thuộc {1, 2, 3, 6, 12}",
    ~payments["installments"].isin([1, 2, 3, 6, 12]), payments)

# Nhất quán
chk("payments", "order_id", "Nhất quán", "order_id có trong orders",
    ~payments["order_id"].isin(orders["order_id"]), payments)
pm = payments.merge(orders[["order_id", "payment_method"]], on="order_id", how="left", suffixes=("", "_orders"))
chk("payments", "payment_method", "Nhất quán", "payment_method khớp orders",
    pm["payment_method"] != pm["payment_method_orders"], pm)

# Chính xác: payment_value = tổng (quantity x unit_price - discount_amount) của đơn
net = (lines["gross"] - lines["discount_amount"]).groupby(lines["order_id"]).sum()
pv = payments.assign(tinh_lai=payments["order_id"].map(net))
pv["lech"] = pv["payment_value"] - pv["tinh_lai"]
chk("payments", "payment_value", "Chính xác", "payment_value = tổng tiền hàng sau giảm của đơn (lệch > 1)",
    pv["lech"].abs() > 1, pv)

# Kịp thời: mọi ngày của sales có dòng traffic
chk("web_traffic", "date", "Kịp thời", "mọi ngày của sales có dòng traffic",
    ~sales["Date"].isin(web_traffic["date"]), sales, nam=sales["Date"].dt.year,
    phan_loai="lỗi thật", xu_ly="đặc trưng traffic để trống + cờ thiếu, hoặc train từ 2013")

xem("web_traffic", "payments")

Khoảng ngày thiếu traffic:

In [ ]:
miss = sales.loc[~sales["Date"].isin(web_traffic["date"]), "Date"]
miss.groupby((miss.diff() != pd.Timedelta("1D")).cumsum()).agg(["min", "max", "size"])

**Nhận xét**

- `payments` đạt điểm tối đa. `payment_value` bằng đúng tổng tiền hàng sau giảm của đơn, không gồm phí ship.
- `web_traffic` thiếu 181 ngày, nhưng đây là **một đoạn liền** 04/07/2012–31/12/2012, không phải lỗ hổng rải rác. Bảng traffic bắt đầu từ 01/01/2013. Từ 2013 trở đi không thiếu ngày nào.
- Xử lý ở Giai đoạn 3: để trống đặc trưng traffic nửa cuối 2012 kèm cờ thiếu, hoặc cho tập train bắt đầu từ 2013.

# Phần 6. shipments, returns, reviews

Kiểm tra:
1. Duy nhất: khoá của từng bảng.
2. Hợp lệ: phí ship ≥ 0; số sao 1–5; lượng trả > 0; lý do trả thuộc danh sách.
3. Nhất quán về khoá, ngày, lượng và trạng thái giữa ba bảng này với `orders`, `order_items`.

In [ ]:
od = orders.set_index("order_id")["order_date"]

# Duy nhất
chk("shipments", "order_id", "Duy nhất", "order_id không trùng", shipments["order_id"].duplicated(keep=False), shipments)
chk("returns", "return_id", "Duy nhất", "return_id không trùng", returns["return_id"].duplicated(keep=False), returns)
chk("reviews", "review_id", "Duy nhất", "review_id không trùng", reviews["review_id"].duplicated(keep=False), reviews)

# Hợp lệ
chk("shipments", "shipping_fee", "Hợp lệ", "shipping_fee >= 0", shipments["shipping_fee"] < 0, shipments)
chk("reviews", "rating", "Hợp lệ", "rating thuộc 1-5", ~reviews["rating"].isin([1, 2, 3, 4, 5]), reviews)
chk("returns", "return_quantity", "Hợp lệ", "return_quantity > 0", returns["return_quantity"] <= 0, returns)
chk("returns", "refund_amount", "Hợp lệ", "refund_amount >= 0", returns["refund_amount"] < 0, returns)
chk("returns", "return_reason", "Hợp lệ", "return_reason thuộc danh sách",
    ~returns["return_reason"].isin(["changed_mind", "defective", "late_delivery", "not_as_described", "wrong_size"]), returns)

# Nhất quán về khoá
for name, df in [("shipments", shipments), ("returns", returns), ("reviews", reviews)]:
    chk(name, "order_id", "Nhất quán", "order_id có trong orders", ~df["order_id"].isin(orders["order_id"]), df)
qty = order_items.groupby(["order_id", "product_id"])["quantity"].sum().rename("qty_mua").reset_index()
rq = returns.merge(qty, on=["order_id", "product_id"], how="left")
chk("returns", ["order_id", "product_id"], "Nhất quán", "(order_id, product_id) có trong order_items", rq["qty_mua"].isna(), rq)
chk("returns", "return_quantity", "Nhất quán", "return_quantity <= quantity đã mua",
    rq["return_quantity"] > rq["qty_mua"], rq)
rv = reviews.merge(orders[["order_id", "customer_id"]], on="order_id", how="left", suffixes=("", "_orders"))
chk("reviews", "customer_id", "Nhất quán", "customer_id khớp orders", rv["customer_id"] != rv["customer_id_orders"], rv)

# Nhất quán về ngày
s = shipments.assign(order_date=shipments["order_id"].map(od))
chk("shipments", "ship_date", "Nhất quán", "ship_date >= order_date", s["ship_date"] < s["order_date"], s)
chk("shipments", "delivery_date", "Nhất quán", "delivery_date >= ship_date", s["delivery_date"] < s["ship_date"], s)
r = returns.assign(order_date=returns["order_id"].map(od))
chk("returns", "return_date", "Nhất quán", "return_date >= order_date", r["return_date"] < r["order_date"], r)
v = reviews.assign(order_date=reviews["order_id"].map(od))
chk("reviews", "review_date", "Nhất quán", "review_date >= order_date", v["review_date"] < v["order_date"], v)

# Nhất quán về trạng thái
sent = orders[orders["order_status"].isin(["shipped", "delivered", "returned"])]
chk("shipments", "order_id", "Nhất quán", "đơn shipped/delivered/returned có vận đơn",
    ~sent["order_id"].isin(shipments["order_id"]), sent, nam=sent["order_date"].dt.year,
    phan_loai="lỗi thật", xu_ly="giữ, không ảnh hưởng doanh thu")
ret = orders[orders["order_status"] == "returned"]
chk("returns", "order_id", "Nhất quán", "đơn returned có dòng trong returns",
    ~ret["order_id"].isin(returns["order_id"]), ret,
    phan_loai="lỗi thật", xu_ly="giữ, không ảnh hưởng doanh thu")

xem("shipments", "returns", "reviews")

Đơn đã gửi nhưng không có vận đơn, theo năm và trạng thái:

In [ ]:
no_ship = sent[~sent["order_id"].isin(shipments["order_id"])]
pd.crosstab(no_ship["order_date"].dt.year, no_ship["order_status"], margins=True)

In [ ]:
# 564 đơn này nằm ở cuối dữ liệu?
print("Khoảng ngày đặt:", no_ship["order_date"].min().date(), "→", no_ship["order_date"].max().date())
print("Ngày cuối của orders:", orders["order_date"].max().date())

**Nhận xét**

- Thứ tự ngày (đặt → giao → trả, đặt → đánh giá), lượng trả, khoá nối đều không có vi phạm.
- **564 đơn** shipped/delivered/returned không có vận đơn, đều đặt trong **10 ngày cuối** (22–31/12/2022). Bảng `shipments` bị cắt sớm hơn `orders`. Đây là lỗi thật kiểu Kịp thời, nhưng không ảnh hưởng doanh thu vì doanh thu lấy từ `order_items`.
- **80 đơn** returned không có dòng trong `returns`. Lỗi thật, cũng không ảnh hưởng doanh thu.

# Phần 7. customers, geography

Kiểm tra:
1. Duy nhất: `customers.customer_id`, `geography.zip`.
2. Hợp lệ: giới tính, nhóm tuổi, kênh thu hút, vùng thuộc danh sách.
3. Nhất quán: `zip` của khách và của đơn có trong `geography`; ngày đặt đơn ≥ ngày đăng ký.

In [ ]:
# Duy nhất
chk("customers", "customer_id", "Duy nhất", "customer_id không trùng",
    customers["customer_id"].duplicated(keep=False), customers)
chk("geography", "zip", "Duy nhất", "zip không trùng", geography["zip"].duplicated(keep=False), geography)

# Đầy đủ
chk("customers", "signup_date", "Đầy đủ", "signup_date không trống", customers["signup_date"].isna(), customers)

# Hợp lệ
chk("customers", "gender", "Hợp lệ", "gender thuộc danh sách",
    ~customers["gender"].isin(["Female", "Male", "Non-binary"]), customers)
chk("customers", "age_group", "Hợp lệ", "age_group thuộc danh sách",
    ~customers["age_group"].isin(["18-24", "25-34", "35-44", "45-54", "55+"]), customers)
chk("customers", "acquisition_channel", "Hợp lệ", "acquisition_channel thuộc danh sách",
    ~customers["acquisition_channel"].isin(["direct", "email_campaign", "organic_search", "paid_search", "referral", "social_media"]), customers)
chk("geography", "region", "Hợp lệ", "region thuộc {Central, East, West}",
    ~geography["region"].isin(["Central", "East", "West"]), geography)

# Nhất quán
chk("customers", "zip", "Nhất quán", "zip có trong geography", ~customers["zip"].isin(geography["zip"]), customers)
chk("orders", "zip", "Nhất quán", "zip có trong geography", ~orders["zip"].isin(geography["zip"]), orders)
oc2 = orders.merge(customers[["customer_id", "signup_date"]], on="customer_id", how="left")
chk("customers", "signup_date", "Nhất quán", "order_date >= signup_date",
    oc2["order_date"] < oc2["signup_date"], oc2, nam=oc2["order_date"].dt.year,
    phan_loai="lỗi thật", xu_ly="bỏ cột signup_date")

xem("customers", "geography")

**Nhận xét**

- `geography` và các cột phân loại của `customers` đều hợp lệ, không trùng, khoá `zip` khớp.
- **73,8% đơn** (477.453) đặt trước ngày khách đăng ký. Tỷ lệ đúng tăng dần theo năm, từ 0,5% (2012) lên 90,9% (2022) (xem Phần 9). Như vậy `signup_date` không phản ánh ngày đăng ký thật. Xử lý: **bỏ cột `signup_date`**, không dùng làm đặc trưng "tuổi khách hàng".

# Phần 8. Bảng điểm "trước"

Ô trống trong bảng điểm nghĩa là bảng đó không có quy tắc nào (tính điểm) ở khía cạnh đó.

In [ ]:
R = pd.DataFrame(RULES)
scored = R[R["Phân loại"] != "đúng nghiệp vụ"]
diem = scored.pivot_table(index="Bảng", columns="Khía cạnh", values="Điểm", aggfunc="mean").reindex(columns=DIMS)
diem["Trung bình"] = diem.mean(axis=1)
diem = diem.round(3).sort_values("Trung bình")

fig, ax = plt.subplots(figsize=(10, 6))
sns.heatmap(diem, annot=True, fmt=".3f", cmap="RdYlGn", vmin=0.5, vmax=1, cbar=False, ax=ax,
            linewidths=0.5, mask=diem.isna())
ax.set_title("Điểm chất lượng dữ liệu trước tiền xử lý (ô trống = không có quy tắc)")
ax.set_xlabel(""); ax.set_ylabel("")
plt.tight_layout(); fig.savefig(OUT / "bang_diem_truoc.png", dpi=150); plt.show()
diem

Danh sách quy tắc có vi phạm, chưa xếp là "đúng nghiệp vụ", xếp theo số dòng vi phạm:

In [ ]:
loi = R[(R["Vi phạm"] > 0) & (R["Phân loại"] != "đúng nghiệp vụ")].sort_values("Vi phạm", ascending=False)
loi[["ID", "Bảng", "Cột", "Khía cạnh", "Quy tắc", "Kiểm tra", "Vi phạm", "Điểm", "Phân loại", "Xử lý ở GĐ3"]]

Quy tắc "đúng nghiệp vụ" (giữ con số để báo cáo, không tính điểm):

In [ ]:
R[R["Phân loại"] == "đúng nghiệp vụ"][["ID", "Bảng", "Cột", "Quy tắc", "Kiểm tra", "Vi phạm", "Xử lý ở GĐ3"]]

**Nhận xét**

- 90 quy tắc: 78 đạt, 8 đúng nghiệp vụ, 4 lỗi thật.
- 11/13 bảng đạt điểm trung bình 1,000. Hai bảng thấp nhất:
  - `customers` 0,908, do Nhất quán 0,631 (`signup_date`);
  - `web_traffic` 0,984, do Kịp thời 0,953 (thiếu nửa cuối 2012).
- `shipments` và `returns` có lỗi thật nhưng chỉ 564 và 80 dòng, nên điểm làm tròn vẫn gần 1.
- Kết luận cho đề tài: các bảng dựng biến mục tiêu (`sales`, `orders`, `order_items`, `products`) **sạch**. Việc của Giai đoạn 3 tập trung vào đặc trưng phụ: bỏ `signup_date`, xử lý traffic 2012, đổi `promo_id` thành cờ.

Không đo (cột đã drop):

In [ ]:
khong_do = pd.DataFrame(SKIPPED, columns=["Bảng", "Cột", "Quy tắc", "Lý do"])
khong_do

# Phần 9. Điểm theo năm

Các quy tắc được gắn năm (tham số `nam=` trong `chk`) được vẽ ở đây. Dùng cho màn hình "theo dõi theo thời gian" của dashboard.

In [ ]:
by = pd.concat(BY_YEAR, ignore_index=True)
fig, ax = plt.subplots(figsize=(10, 5))
for q, g in by.groupby("Quy tắc"):
    ax.plot(g["nam"], g["diem"], marker="o", label=q)
ax.set_ylim(0, 1.05); ax.set_xlabel("Năm"); ax.set_ylabel("Điểm")
ax.set_title("Điểm theo năm của các quy tắc theo dõi")
ax.legend(fontsize=8, loc="lower left"); ax.grid(alpha=0.3)
plt.tight_layout(); fig.savefig(OUT / "diem_theo_nam.png", dpi=150); plt.show()
by.pivot_table(index="nam", columns="Quy tắc", values="diem").round(3)

**Nhận xét**

- `order_date` ≥ `signup_date` tăng đều từ 0,005 (2012) lên 0,909 (2022). Lỗi có hệ thống ở mọi năm, không phải sự cố của một năm.
- Traffic chỉ thiếu năm 2012 (điểm 0), từ 2013 đạt 1.
- Đơn đã gửi có vận đơn chỉ giảm ở 2022 (0,982), do bảng `shipments` bị cắt ở 10 ngày cuối.
- Hai quy tắc "đúng nghiệp vụ" (ngày lỗ gộp, bán dưới giá vốn) dao động 0,75–1,0 theo mức khuyến mãi từng năm, không có xu hướng.

# Phần 10. Lưu kết quả

In [ ]:
R.to_csv(OUT / "dq_rules_truoc.csv", index=False, encoding="utf-8-sig")
diem.to_csv(OUT / "dq_scores_truoc.csv", encoding="utf-8-sig")
khong_do.to_csv(OUT / "dq_khong_do.csv", index=False, encoding="utf-8-sig")
by.to_csv(OUT / "dq_theo_nam.csv", index=False, encoding="utf-8-sig")
print("Đã lưu vào", OUT)
print(R["Phân loại"].value_counts().to_string())